<a href="https://colab.research.google.com/github/shambhavi1709/N-Shot-Classifier/blob/main/FEW_SHOT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# fetch dataset
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((84, 84)),  # few-shot papers use 84x84
])

'''
train_data = [('cat', <tns>), ('flower', <tns>), ('lamp', <tns>), ...] # 50000
test_data =  [('cat', <tns>), ('flower', <tns>), ('lamp', <tns>), ...] # 10000
'''
train_data = datasets.CIFAR100(root="./data", train=True, download=True, transform=transform)
test_data  = datasets.CIFAR100(root="./data", train=False, download=True, transform=transform)

len(train_data), len(test_data)


100%|██████████| 169M/169M [00:10<00:00, 15.6MB/s]


(50000, 10000)

In [2]:
# pivot data
from collections import defaultdict

def pivot_data(dataset):
    class_to_indices = defaultdict(list)
    for tns, label in dataset:
        class_to_indices[label].append(tns)
    return class_to_indices

'''
train_index = {'cat': [<tns>, <tns>], 'dog': [<tns>, <tns>], 'flower': [<tns>, <tns>]}  # 100
test_index =  {'cat': [<tns>, <tns>], 'dog': [<tns>, <tns>], 'flower': [<tns>, <tns>]}  # 100
'''
train_index = pivot_data(train_data)
test_index  = pivot_data(test_data)


In [3]:
# N way K shot Q query
N = 10
K = 5
Q = 2

In [4]:
# Episode - divide each class into Query set and Support set
import random
import torch

def create_episode(data_index, N, K, Q):
  """
  returns:

    query_labels: [0, 0, 0, 1, 1, 1, 2, 2, 2]
    query_images: [<tns>, <tns>, <tns> ...]

    support_labels: [0, 0, 0, 1, 1, 1, 2, 2, 2]
    support_images: [<tns>, <tns>, <tns> ...]
  """
  query_images = []
  query_labels = []

  support_images = []
  support_labels = []

  data_class = random.sample(list(train_index.keys()), N)

  for idx, cls in enumerate(data_class):
      imgs = data_index[cls]
      random_imgs = random.sample(imgs, K + Q)

      query_imgs = random_imgs[:Q]
      support_imgs = random_imgs[Q:]

      for img in query_imgs:
        query_labels.append(idx)
        query_images.append(img)
      for img in support_imgs:
        support_labels.append(idx)
        support_images.append(img)

  support_images = torch.stack(support_images)
  query_images   = torch.stack(query_images)

  support_labels = torch.tensor(support_labels)
  query_labels   = torch.tensor(query_labels)

  return query_labels, query_images, support_labels, support_images

In [5]:
# neural network
import torch.nn as nn
import torch.nn.functional as F

class ProtoEncoder(nn.Module):
  def __init__(self):
    super().__init__()
    # Encoder: each Conv looks for patterns, ReLU keeps positives,
    # MaxPool shrinks the image. This repeats 3 times.
    # Flatten turns the final small feature map into a 1D vector.
    # [64 x 10 x 10]
    self.encoder = nn.Sequential(
        nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),
    )

  def forward(self, x):
        return self.encoder(x)


In [6]:
# create encoder
encoder = ProtoEncoder()
optimizer = torch.optim.Adam(encoder.parameters(), lr=1e-3)

In [7]:
# prototype - mean embedding of a class
def compute_prototypes(emb_support_imgs, support_labels):
  """
  returns:
    prototypes: tensor([N, 6400])
  """
  labels = torch.unique(support_labels)
  prototypes = []

  for label in labels:
    class_vecs = emb_support_imgs[support_labels == label]
    proto = class_vecs.mean(dim=0)
    prototypes.append(proto)

  return torch.stack(prototypes)

In [8]:
def compute_loss(distances, query_labels):
    log_probs = F.log_softmax(-distances, dim=1)
    loss = -log_probs[range(len(query_labels)), query_labels].mean()
    preds = log_probs.argmax(dim=1)
    acc = (preds == query_labels).float().mean()
    return loss, acc

In [9]:
def prototypical_loss(emb_support_img, emb_query_img, support_labels, query_labels):
    prototypes = compute_prototypes(emb_support_img, support_labels)
    distances = torch.cdist(emb_query_img, prototypes)
    return compute_loss(distances, query_labels)


In [ ]:
# TRAIN
for episode in range(2000):  # number of episodes
    # 1. sample episode
    query_labels, query_images, support_labels, support_images = create_episode(
        train_index, N, K, Q
    )

    # 2. encode support + query images
    emb_query_img = encoder(query_images)
    emb_support_img = encoder(support_images)

    # 3. compute loss
    loss, accuracy = prototypical_loss(
        emb_support_img,
        emb_query_img,
        support_labels,
        query_labels,
    )

    # 4. optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if episode % 100 == 0:
        print(f"Episode {episode}, Loss={loss.item():.4f}, Accuracy={accuracy.item():.4f}")

Episode 0, Loss=2.2297, Accuracy=0.3500
Episode 100, Loss=2.3607, Accuracy=0.2000


In [ ]:
# TEST
query_labels, query_images, support_labels, support_images = create_episode(
  test_index, N, K, Q
)

emb_query = encoder(query_images)
emb_support = encoder(support_images)

loss, acc = prototypical_loss(emb_support, emb_query, support_labels, query_labels)

print(f"Final {N}-way {K}-shot accuracy:", acc.item())

In [ ]:
from PIL import Image
from IPython.display import display

def predict_with_distances(encoder, image_path, test_index, N, K, max_display_size=150):
    """
    Predict a single image and show distances to each support class
    """
    # Load query image
    img = Image.open(image_path).convert("RGB")
    display_image = img.copy()
    display_image.thumbnail((max_display_size, max_display_size))

    transform = transforms.Compose([
        transforms.Resize((84, 84)),
        transforms.ToTensor()
    ])
    query_img = transform(img).unsqueeze(0)

    # Sample N support classes and K images per class
    classes = random.sample(list(test_index.keys()), N)

    support_imgs, support_labels = [], []
    for i, cls in enumerate(classes):
        imgs = random.sample(test_index[cls], K)
        for s in imgs:
            support_imgs.append(s)
            support_labels.append(i)

    support_imgs = torch.stack(support_imgs)
    support_labels = torch.tensor(support_labels)

    # Encode
    encoder.eval()
    with torch.no_grad():
        emb_support = encoder(support_imgs)
        emb_query   = encoder(query_img)

        # Compute prototypes
        prototypes = []
        for c in torch.unique(support_labels):
            prototypes.append(emb_support[support_labels == c].mean(dim=0))
        prototypes = torch.stack(prototypes)

        # Compute distances to all prototypes
        dists = torch.cdist(emb_query, prototypes).squeeze(0)  # shape [N]
        pred_idx = dists.argmin().item()
        predicted_class = train_data.classes[classes[pred_idx]]

        # Map distances to class names
        distance_info = {train_data.classes[classes[i]]: dists[i].item() for i in range(N)}

    # Display
    display(display_image)
    print(f"Predicted class: {predicted_class}")
    print("Distances to support classes:")
    for cls_name, dist in distance_info.items():
        print(f"{cls_name}: {dist:.4f}")

    return predicted_class, distance_info

image_path = "./butterfly.jpg"
predicted_class, distances = predict_with_distances(encoder, image_path, test_index, N, K)
